In [1]:
import pandas as pd
import pymysql

conn = None
cur = None

sql = ""

conn = pymysql.connect(host='172.21.27.208', user='root', password='Rkdqnrlte1q', db='Samsung_LTE', charset='utf8')
cur = conn.cursor()

sql = "SELECT * FROM site_info_ru30"
result = pd.read_sql_query(sql,conn)
info = pd.DataFrame(result)

In [2]:
while True:
    start_date = input("날짜를 입력하세요. (ex.20220101)")
    if len(start_date) == 8:
        break
    else:
        print("형식에 맞게 다시 입력해주세요. (ex.20220101)")

while True:
    finish_date = input("끝 날짜를 입력하세요. (ex.20220101)")
    if len(finish_date) == 8:
        break
    else:
        print("형식에 맞게 다시 입력해주세요. (ex.20220101)")

conn = None
cur = None

sql = ""

conn = pymysql.connect(host='172.21.27.208', user='root', password='Rkdqnrlte1q', db='Samsung_LTE', charset='utf8')
cur = conn.cursor()

new_kpi = pd.DataFrame()
for date in pd.date_range(start_date, finish_date):
    input_date = date.strftime('%Y%m%d')
    print(input_date)


    sql = """SELECT c.sday,c.h,c.lsm, c.enb, c.cell, s.rrhid,s.rrhname,s.bw,s.ch, c.prbreal,c.rrc_total,s.pci,s.catype,s.team,s.du_fa_num,s.maxcc,s.coloc,s.auemax,s.auemaxH
FROM (
SELECT a.sday,left(a.stime,2) AS h,a.lsm,a.enb,a.cell, IFNULL(SUM(a.prb_dl_total)/ SUM(a.prb_dl_cnt),0) AS prbreal, SUM(a.rrc_attempt) AS rrc_total
FROM Samsung_LTE.kpi_""" + input_date + """ AS a
WHERE a.stime >= '0900' AND a.stime < '2300'
GROUP BY a.lsm,a.enb,a.cell,h
) AS c
LEFT JOIN Samsung_LTE.site_info AS s ON (c.enb = s.enbname) AND (c.cell = s.cell OR c.cell = s.cell2)"""
    result = pd.read_sql_query(sql,conn)
    kpi = pd.DataFrame(result)

    new_kpi = pd.concat([new_kpi, kpi])
    
print(new_kpi)

20220102
              sday   h   lsm         enb cell       rrhid           rrhname  \
0       2022-01-02  19  0.00        0.00  0.0        None              None   
1       2022-01-02  21  0.00           1    1        None              None   
2       2022-01-02  19     4       31745  403        None              None   
3       2022-01-02  19    67          34   34        None              None   
4       2022-01-02  12  8.54        9.26  0.0        None              None   
...            ...  ..   ...         ...  ...         ...               ...   
496247  2022-01-02  18  LSM9  SEC0010746    9  RNKG80392T  창릉동_삼송현대썬앤빌_900M   
496248  2022-01-02  19  LSM9  SEC0010746    9  RNKG80392T  창릉동_삼송현대썬앤빌_900M   
496249  2022-01-02  20  LSM9  SEC0010746    9  RNKG80392T  창릉동_삼송현대썬앤빌_900M   
496250  2022-01-02  21  LSM9  SEC0010746    9  RNKG80392T  창릉동_삼송현대썬앤빌_900M   
496251  2022-01-02  22  LSM9  SEC0010746    9  RNKG80392T  창릉동_삼송현대썬앤빌_900M   

          bw    ch    prbreal  rrc_total  

In [123]:
join_df = pd.merge(info, new_kpi, left_on = ['eNB_Name', 'Cell'], right_on=['enb', 'cell'], how='inner')

In [152]:
info

,lsms,eNB_Name,eNB_ID,RRH_ID,RRH_Name,KoogSa,Rack,eNB_location,card,port,...,re_port_day,freezone,dl_prb,ul_prb,nb_cell_status,nb_pci,nb_tac,tce_ip,ktx_kyunggang,emtc_switch
0,LSM1,SEC0006169,20685,RNSL03820T,능동_광정빌딩_900M,성수,8,5,0,0,...,20190513,-,23.68,4.45,-,-,-,10.228.79.26,-,12
1,LSM1,SEC0006169,20685,RNSL03819T,군자동_비전빌딩_900M,성수,8,5,0,1,...,20190513,-,23.68,4.45,-,-,-,10.228.79.26,-,12
2,LSM1,SEC0006169,20685,RNSL03839T,중곡2_공영빌딩_900M,성수,8,5,0,2,...,20190513,-,15.87,3.75,-,-,-,10.228.79.26,-,16
3,LSM1,SEC0006169,20685,RNSL80316T,중곡2_공영빌딩2_900M,성수,8,5,0,5,...,20210521,-,99.63,49.15,-,-,-,10.228.79.26,-,18
4,LSM1,SEC0006169,20685,RNSL03832T,중곡3_광덕아파트_900M,성수,8,5,0,6,...,20191011,-,42.35,19.18,-,-,-,10.228.79.26,-,22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36299,YI_LSM5,SEC0009081,-,RRIC03245T,금산ICUHR-ROU3-T,-,-,-,-,-,...,-,-,0.00,0.00,-,-,-,-,KTX,
36300,YI_LSM5,SEC0009081,-,RCIC03245T,금산ICUHR-ROU3-T,-,-,-,-,-,...,-,-,0.00,0.00,-,-,-,-,KTX,
36301,YI_LSM5,SEC0009085,-,RRIC02509T,덕교동전주-W-T,-,-,-,-,-,...,-,-,0.00,0.00,-,-,-,-,KTX,
36302,YI_LSM5,SEC0009085,-,RRIC02511T,덕교동128DPD-W-T,-,-,-,-,-,...,-,-,0.00,0.00,-,-,-,-,KTX,


In [154]:
ru_df = info
ru_df['enb_PCI'] = ru_df['eNB_Name'] + '_' + ru_df['PCI']
ru_df = ru_df[ru_df['BW']=='900M']
ru_df = ru_df.groupby('enb_PCI').count().reset_index()
ru_df['ru'] = ru_df['lsms']
ru_count_df = ru_df[['enb_PCI', 'ru']]
ru_count_df

,enb_PCI,ru
0,SEC0005831_-,1
1,SEC0005833_-,1
2,SEC0005834_-,2
3,SEC0005835_-,4
4,SEC0005861_117,2
...,...,...
6261,SEC9000000_81,1
6262,SECTest_302,1
6263,SECTest_329,1
6264,SECTest_420,1


In [124]:
join_df = join_df[join_df['prbreal'] > 70]

In [125]:
join_df

,lsms,eNB_Name,eNB_ID,RRH_ID,RRH_Name,KoogSa,Rack,eNB_location,card,port,...,prbreal,rrc_total,pci,catype,team_y,du_fa_num,maxcc,coloc,auemax,auemaxH
70,LSM1,SEC0006169,20685,RNSL80316T,중곡2_공영빌딩2_900M,성수,8,5,0,5,...,99.470327,420.0,129,DL_Only,청량,4CA,4,4_7_22,51.00,22
71,LSM1,SEC0006169,20685,RNSL80316T,중곡2_공영빌딩2_900M,성수,8,5,0,5,...,99.462162,479.0,129,DL_Only,청량,4CA,4,4_7_22,51.00,22
72,LSM1,SEC0006169,20685,RNSL80316T,중곡2_공영빌딩2_900M,성수,8,5,0,5,...,99.964708,311.0,129,DL_Only,청량,4CA,4,4_7_22,51.00,22
73,LSM1,SEC0006169,20685,RNSL80316T,중곡2_공영빌딩2_900M,성수,8,5,0,5,...,99.898435,378.0,129,DL_Only,청량,4CA,4,4_7_22,51.00,22
74,LSM1,SEC0006169,20685,RNSL80316T,중곡2_공영빌딩2_900M,성수,8,5,0,5,...,99.717254,334.0,129,DL_Only,청량,4CA,4,4_7_22,51.00,22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
931628,LSM14,SEC0010633,32881,RQKW51701T,속초프리머스T_7번,속초,0,6,1,9,...,71.313756,19236.0,422_422,DL_and_UL_DL_and_UL,영동,3CA,3_3,23_2/5_2,225.00_6.0,16
931629,LSM14,SEC0010633,32881,RQKW51701T,속초프리머스T_7번,속초,0,6,1,9,...,71.313756,19236.0,422_422,DL_and_UL_DL_and_UL,영동,3CA,3_3,23_2/5_2,225.00_6.0,16
933882,LSM14,SEC0010636,32884,RNKW51560T,속초부영_LRIU-T,속초,1,1,0,6,...,74.742409,4097.0,241,DL_and_UL,영동,3CA,3,13_31,68.00,22
934008,LSM14,SEC0010636,32884,RQKW51560T,속초부영_LRIU-T,속초,1,1,1,6,...,72.199105,5786.0,241_241,DL_and_UL_DL_and_UL,영동,3CA,3_3,31_10/13_10,106.00_0.0,00


In [126]:
prepro_df = join_df.pivot_table(['prbreal'], index=['sday', 'h', 'enb','PCI','BW', 'RRH_Name'])
prepro_df = prepro_df.reset_index()
prepro_df['enb_PCI'] = prepro_df['enb'] + "_" + prepro_df['PCI']
prepro_df

,sday,h,enb,PCI,BW,RRH_Name,prbreal,enb_PCI
0,2022-01-02,09,SEC0005866,77,1.8G,장항2_이스턴씨티_RU30,98.091064,SEC0005866_77
1,2022-01-02,09,SEC0005866,77,1.8H,장항2_이스턴씨티_RU30,99.378976,SEC0005866_77
2,2022-01-02,09,SEC0005924,417,1.8G,가회_남북회담본부_1.8G,79.840710,SEC0005924_417
3,2022-01-02,09,SEC0005924,417,1.8G,삼청_감사원_1.8G,79.840710,SEC0005924_417
4,2022-01-02,09,SEC0005924,417,900M,가회_남북회담본부_900M,71.653651,SEC0005924_417
...,...,...,...,...,...,...,...,...
6629,2022-01-02,22,SEC0010813,257,900M,중계23_건영옴니백화점_900M,79.831977,SEC0010813_257
6630,2022-01-02,22,SEC0010813,314,900M,하계1_한신아파트4동_900M,84.818207,SEC0010813_314
6631,2022-01-02,22,SEC0010813,97,900M,하계2_건영벽산굴뚝_900M,87.042261,SEC0010813_97
6632,2022-01-02,22,SEC0440009,263,1.8G,숭인2_숭인동동양파라빌_1.8G,79.768721,SEC0440009_263


In [127]:
grouped = prepro_df.groupby(['enb_PCI', 'BW']).count().reset_index()
grouped2 = prepro_df.groupby(['enb_PCI', 'BW']).max().reset_index()
grouped['count'] = grouped['prbreal']
grouped2['max_prb'] =  grouped2['prbreal']
prb_df = grouped2[['enb_PCI', 'BW', 'max_prb']]
count_df  = grouped[['enb_PCI', 'BW', 'count']]

In [128]:
prepro = pd.pivot_table(prb_df, index=['enb_PCI'], columns='BW', values='max_prb').reset_index()
prepro.dropna(subset=['900M'], inplace=True)

In [130]:
prepro

BW,enb_PCI,1.8E,1.8G,1.8H,2.1E,2.1G,900M
0,SEC0005861_117,0.0,0.000000,0.000000,0.0,0.0,88.069032
1,SEC0005862_157,0.0,80.094234,77.559573,0.0,0.0,73.009575
3,SEC0005862_73,0.0,0.000000,0.000000,0.0,0.0,72.254225
5,SEC0005865_388,0.0,0.000000,0.000000,0.0,0.0,72.230540
6,SEC0005865_48,0.0,0.000000,0.000000,0.0,0.0,70.242034
...,...,...,...,...,...,...,...
927,SEC0010813_257,0.0,0.000000,0.000000,0.0,0.0,84.998520
928,SEC0010813_314,0.0,0.000000,0.000000,0.0,0.0,87.842260
929,SEC0010813_97,0.0,0.000000,0.000000,0.0,0.0,87.042261
930,SEC0440009_263,0.0,86.106611,0.000000,0.0,0.0,83.232885


In [131]:
prepro.fillna(0, inplace=True)

In [132]:
tmp = prepro[['enb_PCI', '900M', '1.8G', '1.8H', '1.8E', '2.1E']]

In [133]:
count_df = count_df[count_df['BW']=='900M']

In [134]:
count_df

,enb_PCI,BW,count
0,SEC0005861_117,900M,4
3,SEC0005862_157,900M,2
5,SEC0005862_73,900M,2
7,SEC0005865_388,900M,2
8,SEC0005865_48,900M,2
...,...,...,...
1207,SEC0010813_257,900M,5
1208,SEC0010813_314,900M,4
1209,SEC0010813_97,900M,5
1211,SEC0440009_263,900M,2


In [135]:
result_count = pd.merge(tmp, count_df[['enb_PCI', 'count']], left_on='enb_PCI', right_on='enb_PCI', how='inner')
result_count.to_csv('./result_count.csv')

In [136]:
result_count['900M'] = round(result_count['900M'], 1)
result_count['1.8G'] = round(result_count['1.8G'], 1)
result_count['1.8H'] = round(result_count['1.8H'], 1)
result_count['1.8E'] = round(result_count['1.8E'], 1)
result_count['2.1E'] = round(result_count['2.1E'], 1)
result_count['900M_Rate'] = round((result_count['900M'] / (result_count['900M'] + result_count['1.8G'] + result_count['1.8H'] + result_count['1.8E'] + result_count['2.1E'])) * 100).astype(int)
result_count['1.8G_Rate'] = round((result_count['1.8G'] / (result_count['900M'] + result_count['1.8G'] + result_count['1.8H'] + result_count['1.8E'] + result_count['2.1E'])) * 100).astype(int)
result_count['1.8H_Rate'] = round((result_count['1.8H'] / (result_count['900M'] + result_count['1.8G'] + result_count['1.8H'] + result_count['1.8E'] + result_count['2.1E'])) * 100).astype(int)
result_count['1.8E_Rate'] = round((result_count['1.8E'] / (result_count['900M'] + result_count['1.8G'] + result_count['1.8H'] + result_count['1.8E'] + result_count['2.1E'])) * 100).astype(int)
result_count['2.1E_Rate'] = round((result_count['2.1E'] / (result_count['900M'] + result_count['1.8G'] + result_count['1.8H'] + result_count['1.8E'] + result_count['2.1E'])) * 100).astype(int)
result_count['900M_Rate'] = result_count['900M_Rate'].astype(str) + '%'
result_count['1.8G_Rate'] = result_count['1.8G_Rate'].astype(str) + '%'
result_count['1.8H_Rate'] = result_count['1.8H_Rate'].astype(str) + '%'
result_count['1.8E_Rate'] = result_count['1.8E_Rate'].astype(str) + '%'
result_count['2.1E_Rate'] = result_count['2.1E_Rate'].astype(str) + '%'

In [137]:
tmp_list = result_count[['900M', '1.8G', '1.8H', '1.8E', '2.1E']]
count_list = []
for i, v  in tmp_list.iterrows():
    count = 0
    for a in v:
        if a != 0:
            count += 1
    count_list.append(count)
result_count['fa'] = count_list
result_count


,enb_PCI,900M,1.8G,1.8H,1.8E,2.1E,count,900M_Rate,1.8G_Rate,1.8H_Rate,1.8E_Rate,2.1E_Rate,fa
0,SEC0005861_117,88.1,0.0,0.0,0.0,0.0,4,100%,0%,0%,0%,0%,1
1,SEC0005862_157,73.0,80.1,77.6,0.0,0.0,2,32%,35%,34%,0%,0%,3
2,SEC0005862_73,72.3,0.0,0.0,0.0,0.0,2,100%,0%,0%,0%,0%,1
3,SEC0005865_388,72.2,0.0,0.0,0.0,0.0,2,100%,0%,0%,0%,0%,1
4,SEC0005865_48,70.2,0.0,0.0,0.0,0.0,2,100%,0%,0%,0%,0%,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
515,SEC0010813_257,85.0,0.0,0.0,0.0,0.0,5,100%,0%,0%,0%,0%,1
516,SEC0010813_314,87.8,0.0,0.0,0.0,0.0,4,100%,0%,0%,0%,0%,1
517,SEC0010813_97,87.0,0.0,0.0,0.0,0.0,5,100%,0%,0%,0%,0%,1
518,SEC0440009_263,83.2,86.1,0.0,0.0,0.0,2,49%,51%,0%,0%,0%,2


In [139]:
prepro_df2 = join_df.pivot_table(['prbreal'], index=['enb','PCI','BW', 'lsm', 'RS_POWER', 'type_MCMC'])
prepro_df2 = prepro_df2.reset_index()
prepro_df2['enb_PCI'] = prepro_df2['enb'] + "_" + prepro_df2['PCI']
prepro_df_900m = prepro_df2[prepro_df2['BW']=='900M']
prepro_df_900m

,enb,PCI,BW,lsm,RS_POWER,type_MCMC,prbreal,enb_PCI
0,SEC0005861,117,900M,LSM5,ci_pwrOpt_Same,MCMC,79.661490,SEC0005861_117
3,SEC0005862,157,900M,LSM5,ci_pwrOpt_Same,MCMC,73.009575,SEC0005862_157
5,SEC0005862,73,900M,LSM5,ci_pwrOpt_Same,MCMC,72.254225,SEC0005862_73
7,SEC0005865,388,900M,LSM5,ci_pwrOpt_Same,MCMC,72.230540,SEC0005865_388
8,SEC0005865,48,900M,LSM5,ci_pwrOpt_Half,MCMC,70.242034,SEC0005865_48
...,...,...,...,...,...,...,...,...
1207,SEC0010813,257,900M,LSM4,ci_pwrOpt_Half,MCMC,77.162409,SEC0010813_257
1208,SEC0010813,314,900M,LSM4,ci_pwrOpt_Half,MCMC,79.435803,SEC0010813_314
1209,SEC0010813,97,900M,LSM4,ci_pwrOpt_Half,MCMC,80.553884,SEC0010813_97
1211,SEC0440009,263,900M,LSM2,ci_pwrOpt_Same,MCMC,77.660170,SEC0440009_263


In [158]:
results = pd.merge(result_count, prepro_df_900m[['enb_PCI', 'lsm', 'RS_POWER', 'type_MCMC']], left_on='enb_PCI', right_on='enb_PCI', how='inner')
results

,enb_PCI,900M,1.8G,1.8H,1.8E,2.1E,count,900M_Rate,1.8G_Rate,1.8H_Rate,1.8E_Rate,2.1E_Rate,fa,lsm,RS_POWER,type_MCMC
0,SEC0005861_117,88.1,0.0,0.0,0.0,0.0,4,100%,0%,0%,0%,0%,1,LSM5,ci_pwrOpt_Same,MCMC
1,SEC0005862_157,73.0,80.1,77.6,0.0,0.0,2,32%,35%,34%,0%,0%,3,LSM5,ci_pwrOpt_Same,MCMC
2,SEC0005862_73,72.3,0.0,0.0,0.0,0.0,2,100%,0%,0%,0%,0%,1,LSM5,ci_pwrOpt_Same,MCMC
3,SEC0005865_388,72.2,0.0,0.0,0.0,0.0,2,100%,0%,0%,0%,0%,1,LSM5,ci_pwrOpt_Same,MCMC
4,SEC0005865_48,70.2,0.0,0.0,0.0,0.0,2,100%,0%,0%,0%,0%,1,LSM5,ci_pwrOpt_Half,MCMC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
515,SEC0010813_257,85.0,0.0,0.0,0.0,0.0,5,100%,0%,0%,0%,0%,1,LSM4,ci_pwrOpt_Half,MCMC
516,SEC0010813_314,87.8,0.0,0.0,0.0,0.0,4,100%,0%,0%,0%,0%,1,LSM4,ci_pwrOpt_Half,MCMC
517,SEC0010813_97,87.0,0.0,0.0,0.0,0.0,5,100%,0%,0%,0%,0%,1,LSM4,ci_pwrOpt_Half,MCMC
518,SEC0440009_263,83.2,86.1,0.0,0.0,0.0,2,49%,51%,0%,0%,0%,2,LSM2,ci_pwrOpt_Same,MCMC


In [160]:
real_result =  pd.merge(results, ru_count_df, left_on='enb_PCI', right_on='enb_PCI', how='inner')
real_result
real_result = real_result[['lsm', 'enb_PCI', '900M', '1.8G', '1.8H', '1.8E', '2.1E', 'fa',
       '900M_Rate', '1.8G_Rate', '1.8H_Rate', '1.8E_Rate', '2.1E_Rate',  'type_MCMC', 'ru', 'RS_POWER', 'count']]

In [164]:
real_result.to_csv('./prb.csv', header=True, index=False)